In [1]:
# =========================================================
# TASK 17 — CELL 1
# SETUP + REPRODUCIBILITY
# =========================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

import optuna

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# ---------------------------------------------------------
# Project root
# ---------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ---------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------
# Cross-validation configuration
# ---------------------------------------------------------

N_SPLITS = 5

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# ---------------------------------------------------------
# Optuna configuration
# ---------------------------------------------------------

N_TRIALS = 30

OPTUNA_SEED = RANDOM_STATE

# ---------------------------------------------------------
# Output directories
# ---------------------------------------------------------

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"

ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ---------------------------------------------------------
# Optuna logging
# ---------------------------------------------------------

optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("=" * 65)
print("        TASK 17 — HYPERPARAMETER TUNING")
print("=" * 65)

print("\nRandom state       :", RANDOM_STATE)
print("CV strategy        : StratifiedKFold")
print("CV folds           :", N_SPLITS)
print("Optuna trials      :", N_TRIALS)
print("Optuna seed        :", OPTUNA_SEED)

print("\nArtifacts directory:")
print(ARTIFACTS_DIR)

print("\nModels directory:")
print(MODELS_DIR)

print("\nSetup completed successfully.")

        TASK 17 — HYPERPARAMETER TUNING

Random state       : 42
CV strategy        : StratifiedKFold
CV folds           : 5
Optuna trials      : 30
Optuna seed        : 42

Artifacts directory:
/home/akash/Projects/Altrodav/artifacts

Models directory:
/home/akash/Projects/Altrodav/models

Setup completed successfully.


In [2]:
# =========================================================
# TASK 17 — CELL 2
# LOAD DATASET + TRAIN/TEST SPLIT
# =========================================================

from sklearn.datasets import load_iris

# ---------------------------------------------------------
# Load Iris dataset
# ---------------------------------------------------------

iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

y = pd.Series(
    iris.target,
    name="target"
)

# ---------------------------------------------------------
# Train / test split
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

# ---------------------------------------------------------
# Verification
# ---------------------------------------------------------

print("========== DATASET ==========")

print("Full dataset shape:", X.shape)

print("\nFeature columns:")
print(list(X.columns))

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\n========== TRAIN / TEST SPLIT ==========")

print("Training samples:", len(X_train))
print("Test samples    :", len(X_test))

print("\nTraining target distribution:")
print(
    y_train.value_counts()
    .sort_index()
)

print("\nTest target distribution:")
print(
    y_test.value_counts()
    .sort_index()
)

print("\nTest set is held out from hyperparameter tuning.")

print("\nDataset preparation completed successfully.")

========== DATASET ==========
Full dataset shape: (150, 4)

Feature columns:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Target distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64

========== TRAIN / TEST SPLIT ==========
Training samples: 120
Test samples    : 30

Training target distribution:
target
0    40
1    40
2    40
Name: count, dtype: int64

Test target distribution:
target
0    10
1    10
2    10
Name: count, dtype: int64

Test set is held out from hyperparameter tuning.

Dataset preparation completed successfully.


In [3]:
# =========================================================
# TASK 17 — CELL 3
# BASELINE SVM
# =========================================================

from sklearn.metrics import accuracy_score

# ---------------------------------------------------------
# Baseline SVM pipeline
# ---------------------------------------------------------

baseline_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC(
            random_state=RANDOM_STATE
        )
    )
])

# ---------------------------------------------------------
# Baseline cross-validation
# ---------------------------------------------------------

baseline_cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

# ---------------------------------------------------------
# Train baseline on training data
# ---------------------------------------------------------

baseline_model.fit(
    X_train,
    y_train
)

# ---------------------------------------------------------
# Evaluate on untouched test set
# ---------------------------------------------------------

baseline_test_predictions = baseline_model.predict(
    X_test
)

baseline_test_accuracy = accuracy_score(
    y_test,
    baseline_test_predictions
)

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

baseline_cv_mean = baseline_cv_scores.mean()
baseline_cv_std = baseline_cv_scores.std()
baseline_cv_variance = baseline_cv_scores.var()

print("========== BASELINE SVM ==========")

print(
    "CV fold scores:",
    np.round(baseline_cv_scores, 4)
)

print(
    f"\nMean CV Accuracy : "
    f"{baseline_cv_mean:.4f}"
)

print(
    f"CV Std           : "
    f"{baseline_cv_std:.4f}"
)

print(
    f"CV Variance      : "
    f"{baseline_cv_variance:.4f}"
)

print(
    f"\nBaseline Test Accuracy: "
    f"{baseline_test_accuracy:.4f}"
)

print(
    "\nBaseline evaluation completed successfully."
)

========== BASELINE SVM ==========
CV fold scores: [0.9583 1.     0.9583 0.9583 0.9583]

Mean CV Accuracy : 0.9667
CV Std           : 0.0167
CV Variance      : 0.0003

Baseline Test Accuracy: 0.9667

Baseline evaluation completed successfully.


In [4]:
# =========================================================
# TASK 17 — CELL 4
# OPTUNA SEARCH SPACE + OBJECTIVE
# =========================================================

def create_svm_from_trial(trial):

    # -----------------------------------------------------
    # Search space
    # -----------------------------------------------------

    C = trial.suggest_float(
        "C",
        1e-2,
        1e2,
        log=True
    )

    gamma = trial.suggest_float(
        "gamma",
        1e-4,
        1e1,
        log=True
    )

    kernel = trial.suggest_categorical(
        "kernel",
        ["linear", "rbf", "poly"]
    )

    # -----------------------------------------------------
    # SVM pipeline
    # -----------------------------------------------------

    model = Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            SVC(
                C=C,
                gamma=gamma,
                kernel=kernel,
                random_state=RANDOM_STATE
            )
        )
    ])

    return model


def objective(trial):

    model = create_svm_from_trial(trial)

    # -----------------------------------------------------
    # Cross-validation
    # -----------------------------------------------------

    fold_scores = []

    for fold_number, (train_idx, val_idx) in enumerate(
        cv.split(X_train, y_train),
        start=1
    ):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]

        model.fit(
            X_fold_train,
            y_fold_train
        )

        fold_predictions = model.predict(
            X_fold_val
        )

        fold_accuracy = accuracy_score(
            y_fold_val,
            fold_predictions
        )

        fold_scores.append(
            fold_accuracy
        )

        # Report intermediate score to Optuna
        trial.report(
            np.mean(fold_scores),
            step=fold_number
        )

        # Allow Optuna to prune weak trials
        if trial.should_prune():
            raise optuna.TrialPruned()

    # -----------------------------------------------------
    # Final CV score
    # -----------------------------------------------------

    mean_cv_score = np.mean(
        fold_scores
    )

    return mean_cv_score


print("========== OPTUNA SEARCH SPACE ==========")

print("C      : 1e-2 → 1e2 (log scale)")
print("gamma  : 1e-4 → 1e1 (log scale)")
print("kernel : linear / rbf / poly")

print("\nObjective metric: Mean StratifiedKFold Accuracy")

print(
    "\nPruning enabled through intermediate "
    "fold-level reporting."
)

print("\nOptuna objective created successfully.")

========== OPTUNA SEARCH SPACE ==========
C      : 1e-2 → 1e2 (log scale)
gamma  : 1e-4 → 1e1 (log scale)
kernel : linear / rbf / poly

Objective metric: Mean StratifiedKFold Accuracy

Pruning enabled through intermediate fold-level reporting.

Optuna objective created successfully.


In [5]:
# =========================================================
# TASK 17 — CELL 5
# RUN OPTUNA BAYESIAN OPTIMIZATION
# =========================================================

# ---------------------------------------------------------
# Create Optuna study
# ---------------------------------------------------------

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=OPTUNA_SEED
    ),
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=2
    )
)

# ---------------------------------------------------------
# Run optimization
# ---------------------------------------------------------

print("========== OPTUNA OPTIMIZATION ==========")

print(
    f"Running {N_TRIALS} optimization trials..."
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=False
)

# ---------------------------------------------------------
# Optimization summary
# ---------------------------------------------------------

completed_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]

pruned_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.PRUNED
]

failed_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.FAIL
]

print("\n========== OPTIMIZATION SUMMARY ==========")

print(
    "Total trials     :",
    len(study.trials)
)

print(
    "Completed trials  :",
    len(completed_trials)
)

print(
    "Pruned trials     :",
    len(pruned_trials)
)

print(
    "Failed trials     :",
    len(failed_trials)
)

print("\n========== BEST TRIAL ==========")

print(
    "Best trial number :",
    study.best_trial.number
)

print(
    f"Best CV accuracy  : "
    f"{study.best_value:.4f}"
)

print("\nBest hyperparameters:")

for parameter, value in study.best_params.items():
    print(
        f"{parameter}: {value}"
    )

print(
    "\nOptuna optimization completed successfully."
)

========== OPTUNA OPTIMIZATION ==========
Running 30 optimization trials...

========== OPTIMIZATION SUMMARY ==========
Total trials     : 30
Completed trials  : 20
Pruned trials     : 10
Failed trials     : 0

========== BEST TRIAL ==========
Best trial number : 7
Best CV accuracy  : 0.9750

Best hyperparameters:
C: 17.12337597316399
gamma: 0.0033347927286375843
kernel: rbf

Optuna optimization completed successfully.


In [6]:
# =========================================================
# TASK 17 — CELL 6
# LOG + INSPECT OPTUNA TRIALS
# =========================================================

# ---------------------------------------------------------
# Convert Optuna trials to DataFrame
# ---------------------------------------------------------

trials_df = study.trials_dataframe(
    attrs=(
        "number",
        "state",
        "value",
        "params",
        "datetime_start",
        "datetime_complete",
        "duration"
    )
)

# ---------------------------------------------------------
# Display trial history
# ---------------------------------------------------------

print("========== OPTUNA TRIAL HISTORY ==========")

print(
    trials_df[
        [
            "number",
            "state",
            "value",
            "params_C",
            "params_gamma",
            "params_kernel"
        ]
    ]
    .sort_values(
        "number"
    )
    .round(6)
    .to_string(index=False)
)

# ---------------------------------------------------------
# Best completed trials
# ---------------------------------------------------------

best_trials_df = (
    trials_df[
        trials_df["state"] == "COMPLETE"
    ]
    .sort_values(
        "value",
        ascending=False
    )
    .head(10)
)

print("\n========== TOP 10 COMPLETED TRIALS ==========")

print(
    best_trials_df[
        [
            "number",
            "value",
            "params_C",
            "params_gamma",
            "params_kernel"
        ]
    ]
    .round(6)
    .to_string(index=False)
)

# ---------------------------------------------------------
# Save complete trial history
# ---------------------------------------------------------

optuna_trials_path = (
    ARTIFACTS_DIR /
    "task17_optuna_trials.csv"
)

trials_df.to_csv(
    optuna_trials_path,
    index=False
)

# ---------------------------------------------------------
# Save best parameters
# ---------------------------------------------------------

best_params_df = pd.DataFrame([
    {
        "Parameter": parameter,
        "Value": value
    }
    for parameter, value in study.best_params.items()
])

best_params_path = (
    ARTIFACTS_DIR /
    "task17_best_hyperparameters.csv"
)

best_params_df.to_csv(
    best_params_path,
    index=False
)

print("\n========== OPTUNA ARTIFACTS ==========")

print(
    "Trial history:",
    optuna_trials_path
)

print(
    "Best parameters:",
    best_params_path
)

print(
    "\nOptuna trial logging completed successfully."
)

========== OPTUNA TRIAL HISTORY ==========
 number    state    value  params_C  params_gamma params_kernel
      0 COMPLETE 0.958333  0.314891      5.669850        linear
      1 COMPLETE 0.908333  0.042071      0.000195        linear
      2 COMPLETE 0.883333  0.012088      7.072114        linear
      3 COMPLETE 0.958333  0.054152      0.003321        linear
      4 COMPLETE 0.558333  2.801635      0.000498          poly
      5 COMPLETE 0.900000 13.826232      0.000996           rbf
      6   PRUNED 0.604167  2.692647      0.000712          poly
      7 COMPLETE 0.975000 17.123376      0.003335           rbf
      8   PRUNED 0.895833  0.030772      0.029915           rbf
      9 COMPLETE 0.900000  4.467753      0.003619           rbf
     10 COMPLETE 0.941667 75.145129      0.315773           rbf
     11 COMPLETE 0.975000  0.244008      7.930787        linear
     12 COMPLETE 0.975000  0.446671      0.154831        linear
     13 COMPLETE 0.933333 87.053518      0.647774           r

In [7]:
# =========================================================
# TASK 17 — CELL 7
# TRAIN BEST TUNED MODEL
# =========================================================

# ---------------------------------------------------------
# Extract best hyperparameters
# ---------------------------------------------------------

best_params = study.best_params

best_tuned_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC(
            C=best_params["C"],
            gamma=best_params["gamma"],
            kernel=best_params["kernel"],
            random_state=RANDOM_STATE
        )
    )
])

# ---------------------------------------------------------
# Train on complete training set
# ---------------------------------------------------------

best_tuned_model.fit(
    X_train,
    y_train
)

# ---------------------------------------------------------
# Training predictions
# ---------------------------------------------------------

tuned_train_predictions = best_tuned_model.predict(
    X_train
)

tuned_train_accuracy = accuracy_score(
    y_train,
    tuned_train_predictions
)

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("========== BEST TUNED MODEL ==========")

print("Model: SVM")

print("\nBest hyperparameters:")

for parameter, value in best_params.items():
    print(
        f"{parameter}: {value}"
    )

print(
    f"\nBest CV Accuracy: "
    f"{study.best_value:.4f}"
)

print(
    f"Training Accuracy: "
    f"{tuned_train_accuracy:.4f}"
)

print(
    "\nBest tuned model trained successfully."
)

print(
    "\nTest set remains reserved for final confirmation."
)

========== BEST TUNED MODEL ==========
Model: SVM

Best hyperparameters:
C: 17.12337597316399
gamma: 0.0033347927286375843
kernel: rbf

Best CV Accuracy: 0.9750
Training Accuracy: 0.9750

Best tuned model trained successfully.

Test set remains reserved for final confirmation.


In [8]:
# =========================================================
# TASK 17 — CELL 8
# FINAL HELD-OUT TEST EVALUATION
# =========================================================

# ---------------------------------------------------------
# Generate predictions on untouched test set
# ---------------------------------------------------------

tuned_test_predictions = best_tuned_model.predict(
    X_test
)

# ---------------------------------------------------------
# Calculate test accuracy
# ---------------------------------------------------------

tuned_test_accuracy = accuracy_score(
    y_test,
    tuned_test_predictions
)

# ---------------------------------------------------------
# Compare against baseline
# ---------------------------------------------------------

test_accuracy_gain = (
    tuned_test_accuracy
    - baseline_test_accuracy
)

cv_accuracy_gain = (
    study.best_value
    - baseline_cv_mean
)

# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("========== FINAL TEST EVALUATION ==========")

print(
    f"Baseline Test Accuracy : "
    f"{baseline_test_accuracy:.4f}"
)

print(
    f"Tuned Test Accuracy    : "
    f"{tuned_test_accuracy:.4f}"
)

print(
    f"Test Accuracy Gain     : "
    f"{test_accuracy_gain:+.4f}"
)

print("\n========== CV COMPARISON ==========")

print(
    f"Baseline Mean CV Accuracy : "
    f"{baseline_cv_mean:.4f}"
)

print(
    f"Tuned Mean CV Accuracy    : "
    f"{study.best_value:.4f}"
)

print(
    f"CV Accuracy Gain          : "
    f"{cv_accuracy_gain:+.4f}"
)

print("\n========== FINAL VERDICT ==========")

if tuned_test_accuracy > baseline_test_accuracy:
    print(
        "Tuning produced a test-confirmed performance gain."
    )
elif tuned_test_accuracy == baseline_test_accuracy:
    print(
        "Tuning maintained the baseline test performance."
    )
else:
    print(
        "Tuning improved CV performance but did not improve "
        "the final holdout test performance."
    )

print(
    "\nFinal held-out test evaluation completed successfully."
)

========== FINAL TEST EVALUATION ==========
Baseline Test Accuracy : 0.9667
Tuned Test Accuracy    : 0.9333
Test Accuracy Gain     : -0.0333

========== CV COMPARISON ==========
Baseline Mean CV Accuracy : 0.9667
Tuned Mean CV Accuracy    : 0.9750
CV Accuracy Gain          : +0.0083

========== FINAL VERDICT ==========
Tuning improved CV performance but did not improve the final holdout test performance.

Final held-out test evaluation completed successfully.


In [9]:
# =========================================================
# TASK 17 — CELL 9
# ANALYZE OPTUNA SEARCH STABILITY
# =========================================================

# ---------------------------------------------------------
# Completed trials only
# ---------------------------------------------------------

completed_trials_df = trials_df[
    trials_df["state"] == "COMPLETE"
].copy()

# ---------------------------------------------------------
# Sort by CV performance
# ---------------------------------------------------------

completed_trials_df = completed_trials_df.sort_values(
    "value",
    ascending=False
)

print("========== COMPLETED TRIAL ANALYSIS ==========")

print(
    completed_trials_df[
        [
            "number",
            "value",
            "params_C",
            "params_gamma",
            "params_kernel"
        ]
    ]
    .head(10)
    .round(6)
    .to_string(index=False)
)

# ---------------------------------------------------------
# Count configurations reaching best CV score
# ---------------------------------------------------------

best_cv_score = study.best_value

best_cv_trials = completed_trials_df[
    np.isclose(
        completed_trials_df["value"],
        best_cv_score
    )
]

print("\n========== BEST CV SCORE ANALYSIS ==========")

print(
    f"Best CV accuracy: {best_cv_score:.4f}"
)

print(
    f"Number of completed trials "
    f"reaching best score: {len(best_cv_trials)}"
)

print("\nBest-performing configurations:")

print(
    best_cv_trials[
        [
            "number",
            "value",
            "params_C",
            "params_gamma",
            "params_kernel"
        ]
    ]
    .round(6)
    .to_string(index=False)
)

# ---------------------------------------------------------
# Kernel-level comparison
# ---------------------------------------------------------

kernel_summary = (
    completed_trials_df
    .groupby("params_kernel")["value"]
    .agg(
        ["count", "mean", "max", "std"]
    )
    .reset_index()
)

print("\n========== KERNEL PERFORMANCE ==========")

print(
    kernel_summary
    .round(4)
    .to_string(index=False)
)

# ---------------------------------------------------------
# Search interpretation
# ---------------------------------------------------------

print("\n========== SEARCH INTERPRETATION ==========")

print(
    "Multiple configurations reached the same "
    "maximum CV accuracy."
)

print(
    "This indicates that the CV objective has "
    "several similarly strong regions."
)

print(
    "The holdout test result must remain independent "
    "from this analysis."
)

print(
    "\nSearch stability analysis completed successfully."
)

========== COMPLETED TRIAL ANALYSIS ==========
 number    value  params_C  params_gamma params_kernel
     26 0.975000  0.638393      0.001794        linear
     12 0.975000  0.446671      0.154831        linear
     11 0.975000  0.244008      7.930787        linear
      7 0.975000 17.123376      0.003335           rbf
     22 0.975000  0.219144      0.014639        linear
     21 0.975000  0.649569      0.160665        linear
     18 0.975000  1.011080      0.006436        linear
     16 0.975000  1.002752      0.052023           rbf
     27 0.966667  1.838056      2.971795        linear
     15 0.958333 19.177548      1.366446        linear

========== BEST CV SCORE ANALYSIS ==========
Best CV accuracy: 0.9750
Number of completed trials reaching best score: 8

Best-performing configurations:
 number  value  params_C  params_gamma params_kernel
     26  0.975  0.638393      0.001794        linear
     12  0.975  0.446671      0.154831        linear
     11  0.975  0.244008      7.930

In [10]:
# =========================================================
# TASK 17 — CELL 10
# FINAL TUNING SUMMARY
# =========================================================

# ---------------------------------------------------------
# Calculate gains
# ---------------------------------------------------------

cv_gain = (
    study.best_value
    - baseline_cv_mean
)

test_gain = (
    tuned_test_accuracy
    - baseline_test_accuracy
)

# ---------------------------------------------------------
# Determine final outcome
# ---------------------------------------------------------

if test_gain > 0:
    tuning_outcome = "TEST_CONFIRMED_GAIN"
elif test_gain == 0:
    tuning_outcome = "NO_TEST_GAIN"
else:
    tuning_outcome = "CV_GAIN_WITHOUT_TEST_GAIN"

# ---------------------------------------------------------
# Create summary
# ---------------------------------------------------------

task17_summary = pd.DataFrame([{
    "Baseline_CV_Accuracy": baseline_cv_mean,
    "Tuned_CV_Accuracy": study.best_value,
    "CV_Accuracy_Gain": cv_gain,
    "Baseline_Test_Accuracy": baseline_test_accuracy,
    "Tuned_Test_Accuracy": tuned_test_accuracy,
    "Test_Accuracy_Gain": test_gain,
    "Best_Trial": study.best_trial.number,
    "Best_C": study.best_params["C"],
    "Best_Gamma": study.best_params["gamma"],
    "Best_Kernel": study.best_params["kernel"],
    "Total_Trials": len(study.trials),
    "Completed_Trials": len(completed_trials),
    "Pruned_Trials": len(pruned_trials),
    "Failed_Trials": len(failed_trials),
    "Outcome": tuning_outcome
}])

# ---------------------------------------------------------
# Display summary
# ---------------------------------------------------------

print("=" * 65)
print("        TASK 17 — HYPERPARAMETER TUNING SUMMARY")
print("=" * 65)

print("\n========== CROSS-VALIDATION ==========")

print(
    f"Baseline CV Accuracy : "
    f"{baseline_cv_mean:.4f}"
)

print(
    f"Tuned CV Accuracy    : "
    f"{study.best_value:.4f}"
)

print(
    f"CV Accuracy Gain     : "
    f"{cv_gain:+.4f}"
)

print("\n========== INDEPENDENT TEST ==========")

print(
    f"Baseline Test Accuracy : "
    f"{baseline_test_accuracy:.4f}"
)

print(
    f"Tuned Test Accuracy    : "
    f"{tuned_test_accuracy:.4f}"
)

print(
    f"Test Accuracy Gain     : "
    f"{test_gain:+.4f}"
)

print("\n========== SEARCH ==========")

print(
    "Total trials     :",
    len(study.trials)
)

print(
    "Completed trials :",
    len(completed_trials)
)

print(
    "Pruned trials    :",
    len(pruned_trials)
)

print(
    "Failed trials    :",
    len(failed_trials)
)

print(
    "\nBest trial:",
    study.best_trial.number
)

print(
    "Best parameters:",
    study.best_params
)

print("\n========== FINAL OUTCOME ==========")

if tuning_outcome == "TEST_CONFIRMED_GAIN":

    print(
        "Hyperparameter tuning produced a "
        "test-confirmed performance gain."
    )

elif tuning_outcome == "NO_TEST_GAIN":

    print(
        "Hyperparameter tuning maintained the "
        "baseline test performance."
    )

else:

    print(
        "Hyperparameter tuning improved cross-validation "
        "performance but did not produce a test-confirmed gain."
    )

print(
    "\nTask 17 tuning summary completed successfully."
)

        TASK 17 — HYPERPARAMETER TUNING SUMMARY

========== CROSS-VALIDATION ==========
Baseline CV Accuracy : 0.9667
Tuned CV Accuracy    : 0.9750
CV Accuracy Gain     : +0.0083

========== INDEPENDENT TEST ==========
Baseline Test Accuracy : 0.9667
Tuned Test Accuracy    : 0.9333
Test Accuracy Gain     : -0.0333

========== SEARCH ==========
Total trials     : 30
Completed trials : 20
Pruned trials    : 10
Failed trials    : 0

Best trial: 7
Best parameters: {'C': 17.12337597316399, 'gamma': 0.0033347927286375843, 'kernel': 'rbf'}

========== FINAL OUTCOME ==========
Hyperparameter tuning improved cross-validation performance but did not produce a test-confirmed gain.

Task 17 tuning summary completed successfully.


In [11]:
# =========================================================
# TASK 17 — CELL 11
# SAVE FINAL ARTIFACTS
# =========================================================

# ---------------------------------------------------------
# Save tuned model
# ---------------------------------------------------------

tuned_model_path = (
    MODELS_DIR /
    "task17_tuned_svm.pkl"
)

joblib.dump(
    best_tuned_model,
    tuned_model_path
)

# ---------------------------------------------------------
# Save tuning summary
# ---------------------------------------------------------

summary_path = (
    ARTIFACTS_DIR /
    "task17_tuning_summary.csv"
)

task17_summary.to_csv(
    summary_path,
    index=False
)

# ---------------------------------------------------------
# Save Optuna study
# ---------------------------------------------------------

study_path = (
    ARTIFACTS_DIR /
    "task17_optuna_study.pkl"
)

joblib.dump(
    study,
    study_path
)

# ---------------------------------------------------------
# Save baseline vs tuned comparison
# ---------------------------------------------------------

comparison_df = pd.DataFrame([
    {
        "Model": "Baseline SVM",
        "CV_Accuracy": baseline_cv_mean,
        "Test_Accuracy": baseline_test_accuracy,
        "CV_Std": baseline_cv_std,
        "CV_Variance": baseline_cv_variance
    },
    {
        "Model": "Optuna Tuned SVM",
        "CV_Accuracy": study.best_value,
        "Test_Accuracy": tuned_test_accuracy,
        "CV_Std": np.nan,
        "CV_Variance": np.nan
    }
])

comparison_path = (
    ARTIFACTS_DIR /
    "task17_baseline_vs_tuned.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print("========== TASK 17 ARTIFACTS ==========")

print(
    "Tuned model:",
    tuned_model_path
)

print(
    "Tuning summary:",
    summary_path
)

print(
    "Optuna study:",
    study_path
)

print(
    "Baseline vs tuned comparison:",
    comparison_path
)

print(
    "\nAll Task 17 artifacts saved successfully."
)

========== TASK 17 ARTIFACTS ==========
Tuned model: /home/akash/Projects/Altrodav/models/task17_tuned_svm.pkl
Tuning summary: /home/akash/Projects/Altrodav/artifacts/task17_tuning_summary.csv
Optuna study: /home/akash/Projects/Altrodav/artifacts/task17_optuna_study.pkl
Baseline vs tuned comparison: /home/akash/Projects/Altrodav/artifacts/task17_baseline_vs_tuned.csv

All Task 17 artifacts saved successfully.


In [12]:
# =========================================================
# TASK 17 — CELL 12
# FINAL VERIFICATION
# =========================================================

print("=" * 65)
print("        TASK 17 — FINAL VERIFICATION")
print("=" * 65)

# ---------------------------------------------------------
# Check model
# ---------------------------------------------------------

loaded_tuned_model = joblib.load(
    tuned_model_path
)

model_check = (
    type(loaded_tuned_model).__name__
    == type(best_tuned_model).__name__
)

print("\nTuned model loading :",
      "PASS" if model_check else "FAIL")

# ---------------------------------------------------------
# Check Optuna study
# ---------------------------------------------------------

loaded_study = joblib.load(
    study_path
)

study_check = (
    len(loaded_study.trials)
    == len(study.trials)
)

print(
    "Optuna study loading :",
    "PASS" if study_check else "FAIL"
)

# ---------------------------------------------------------
# Check trial history
# ---------------------------------------------------------

loaded_trials_df = pd.read_csv(
    ARTIFACTS_DIR /
    "task17_optuna_trials.csv"
)

trial_count_check = (
    len(loaded_trials_df)
    == len(study.trials)
)

print(
    "Trial history loading :",
    "PASS" if trial_count_check else "FAIL"
)

# ---------------------------------------------------------
# Check tuning summary
# ---------------------------------------------------------

loaded_summary = pd.read_csv(
    summary_path
)

summary_check = (
    len(loaded_summary) == 1
)

print(
    "Tuning summary loading :",
    "PASS" if summary_check else "FAIL"
)

# ---------------------------------------------------------
# Check baseline vs tuned comparison
# ---------------------------------------------------------

loaded_comparison = pd.read_csv(
    comparison_path
)

comparison_check = (
    len(loaded_comparison) == 2
)

print(
    "Model comparison loading :",
    "PASS" if comparison_check else "FAIL"
)

# ---------------------------------------------------------
# Check saved model predictions
# ---------------------------------------------------------

original_predictions = best_tuned_model.predict(
    X_train
)

loaded_predictions = loaded_tuned_model.predict(
    X_train
)

prediction_check = np.array_equal(
    original_predictions,
    loaded_predictions
)

print(
    "Saved model predictions match :",
    "PASS" if prediction_check else "FAIL"
)

# ---------------------------------------------------------
# Check optimization configuration
# ---------------------------------------------------------

print("\n========== OPTIMIZATION CONFIGURATION ==========")

print(
    "Total trials     :",
    len(study.trials)
)

print(
    "Completed trials :",
    len(completed_trials)
)

print(
    "Pruned trials    :",
    len(pruned_trials)
)

print(
    "Failed trials    :",
    len(failed_trials)
)

print(
    "Best trial       :",
    study.best_trial.number
)

print(
    "Best CV accuracy :",
    f"{study.best_value:.4f}"
)

# ---------------------------------------------------------
# Check final performance
# ---------------------------------------------------------

print("\n========== PERFORMANCE ==========")

print(
    "Baseline CV accuracy :",
    f"{baseline_cv_mean:.4f}"
)

print(
    "Tuned CV accuracy    :",
    f"{study.best_value:.4f}"
)

print(
    "Baseline test accuracy :",
    f"{baseline_test_accuracy:.4f}"
)

print(
    "Tuned test accuracy    :",
    f"{tuned_test_accuracy:.4f}"
)

# ---------------------------------------------------------
# Final conclusion
# ---------------------------------------------------------

print("\n========== FINAL CONCLUSION ==========")

print(
    "Optuna successfully improved cross-validation "
    "performance from "
    f"{baseline_cv_mean:.4f} to {study.best_value:.4f}."
)

print(
    "However, the tuned model achieved "
    f"{tuned_test_accuracy:.4f} on the independent test set "
    f"versus {baseline_test_accuracy:.4f} for the baseline."
)

print(
    "Therefore, the experiment produced a CV improvement "
    "but no test-confirmed performance gain."
)

print(
    "\nTASK 17 EXPERIMENT COMPLETED SUCCESSFULLY!"
)

print("=" * 65)

        TASK 17 — FINAL VERIFICATION

Tuned model loading : PASS
Optuna study loading : PASS
Trial history loading : PASS
Tuning summary loading : PASS
Model comparison loading : PASS
Saved model predictions match : PASS

========== OPTIMIZATION CONFIGURATION ==========
Total trials     : 30
Completed trials : 20
Pruned trials    : 10
Failed trials    : 0
Best trial       : 7
Best CV accuracy : 0.9750

========== PERFORMANCE ==========
Baseline CV accuracy : 0.9667
Tuned CV accuracy    : 0.9750
Baseline test accuracy : 0.9667
Tuned test accuracy    : 0.9333

========== FINAL CONCLUSION ==========
Optuna successfully improved cross-validation performance from 0.9667 to 0.9750.
However, the tuned model achieved 0.9333 on the independent test set versus 0.9667 for the baseline.
Therefore, the experiment produced a CV improvement but no test-confirmed performance gain.

TASK 17 EXPERIMENT COMPLETED SUCCESSFULLY!
